In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval={interval}&range={range}

AAPL
GME

EURUSD=X
USDTRY=X

BTC-USD
DOGE-USD

In [ ]:
import requests

url = "https://query1.finance.yahoo.com/v8/finance/chart/AAPL"
params = {
    "interval": "1d",
    "range": "2y",
}
headers = {
    # Yahoo blocks requests with no User-Agent header, treating them as bots.
    # We just need to look like a normal browser.
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(url, params=params, headers=headers)
print("Status code:", response.status_code)

data = response.json()
data

In [5]:
result = data["chart"]["result"][0]

timestamps = result["timestamp"]
quote = result["indicators"]["quote"][0]

print("Number of candles returned:", len(timestamps))
print("First timestamp (raw):", timestamps[0])
print("Open prices, first 5:", quote["open"][:5])
print("Close prices, first 5:", quote["close"][:5])

Number of candles returned: 501
First timestamp (raw): 1721655000
Open prices, first 5: [227.00999450683594, 224.3699951171875, 224.0, 218.92999267578125, 218.6999969482422]
Close prices, first 5: [223.9600067138672, 225.00999450683594, 218.5399932861328, 217.49000549316406, 217.9600067138672]


In [6]:
from datetime import datetime, timezone

first_date = datetime.fromtimestamp(timestamps[0], tz=timezone.utc)
print("First candle date:", first_date)

First candle date: 2024-07-22 13:30:00+00:00


In [7]:
def parse_yahoo_response(raw_json, symbol):
    """Convert a raw Yahoo chart API response into our OHLCV candle format:
    a list of dicts, each with timestamp, open, high, low, close, volume.
    """
    result = raw_json["chart"]["result"]

    if result is None:
        # Yahoo returns this when the symbol doesn't exist or has no data.
        raise ValueError(f"No data returned for symbol: {symbol}")

    result = result[0]
    timestamps = result["timestamp"]
    quote = result["indicators"]["quote"][0]

    candles = []
    for i in range(len(timestamps)):
        o = quote["open"][i]
        h = quote["high"][i]
        l = quote["low"][i]
        c = quote["close"][i]
        v = quote["volume"][i]

        # Yahoo sometimes returns None for a field -- holidays, thin liquidity,
        # or just gaps in their data. We skip incomplete candles for now.
        # Video 5 is entirely dedicated to handling this properly and honestly
        # reporting how much data we had to throw away.
        if None in (o, h, l, c):
            continue

        candles.append({
            "timestamp": timestamps[i],
            "open": o,
            "high": h,
            "low": l,
            "close": c,
            "volume": v if v is not None else 0,
        })

    return candles


candles = parse_yahoo_response(data, "AAPL")
print(f"Parsed {len(candles)} clean candles out of {len(timestamps)} returned")
print("First candle:", candles[0])
print("Last candle:", candles[-1])

Parsed 501 clean candles out of 501 returned
First candle: {'timestamp': 1721655000, 'open': 227.00999450683594, 'high': 227.77999877929688, 'low': 223.08999633789062, 'close': 223.9600067138672, 'volume': 48201800}
Last candle: {'timestamp': 1784640600, 'open': 323.3450012207031, 'high': 329.6000061035156, 'low': 322.22039794921875, 'close': 327.80499267578125, 'volume': 27523773}


In [9]:
import time

def fetch_ohlcv(symbol, interval="1d", range_="2y"):
    """Fetch OHLCV candles for a single symbol from Yahoo Finance.

    Returns a list of candle dicts, or an empty list if the fetch failed.
    Prints what happened either way -- we report honestly, even on failure.
    """
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    params = {"interval": interval, "range": range_}
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        raw_json = response.json()
        candles = parse_yahoo_response(raw_json, symbol)
        print(f"✅ {symbol}: fetched {len(candles)} candles")
        return candles

    except Exception as e:
        print(f"❌ {symbol}: failed to fetch -- {e}")
        return []

    finally:
        # Be a polite guest. A short pause between requests avoids hammering
        # a free, unofficial endpoint that could rate-limit or block us.
        time.sleep(1)

In [16]:
universe = {
    "forex": [
        # majors
        "EURUSD=X", "GBPUSD=X", "USDJPY=X", "USDCHF=X", "USDCAD=X",
        "AUDUSD=X", "NZDUSD=X", "EURGBP=X", "EURJPY=X", "GBPJPY=X",
        # exotics
        "USDTRY=X", "USDZAR=X", "USDMXN=X", "USDSEK=X", "USDNOK=X",
        "USDHUF=X", "USDPLN=X", "USDTHB=X", "USDINR=X", "USDBRL=X",
    ],
    "crypto": [
        # majors
        "BTC-USD", "ETH-USD", "BNB-USD", "XRP-USD", "SOL-USD",
        "ADA-USD", "AVAX-USD", "DOT-USD", "LINK-USD", "LTC-USD",
        # exotics
        "DOGE-USD", "SHIB-USD", "TRX-USD", "MATIC-USD", "XLM-USD",
        "ALGO-USD", "ETC-USD", "EOS-USD", "XMR-USD", "ATOM-USD",
    ],
    "stocks": [
        # majors
        "AAPL", "MSFT", "GOOGL", "AMZN", "JPM",
        "JNJ", "PG", "KO", "WMT", "V",
        # exotics
        "GME", "AMC", "TSLA", "COIN", "MSTR",
        "PLTR", "RIVN", "PLUG", "SNDL", "BB",
    ],
}

for market, symbols in universe.items():
    print(f"{market}: {len(symbols)} symbols")

print("Total symbols:", sum(len(s) for s in universe.values()))

forex: 20 symbols
crypto: 20 symbols
stocks: 20 symbols
Total symbols: 60


In [17]:
all_data = {}  # e.g. all_data["forex"]["EURUSD=X"] = [list of candles]

for market, symbols in universe.items():
    all_data[market] = {}
    print(f"\n--- Fetching {market} ---")
    for symbol in symbols:
        candles = fetch_ohlcv(symbol, interval="1d", range_="2y")
        all_data[market][symbol] = candles


--- Fetching forex ---
✅ EURUSD=X: fetched 517 candles
✅ GBPUSD=X: fetched 517 candles
✅ USDJPY=X: fetched 517 candles
✅ USDCHF=X: fetched 517 candles
✅ USDCAD=X: fetched 517 candles
✅ AUDUSD=X: fetched 517 candles
✅ NZDUSD=X: fetched 517 candles
✅ EURGBP=X: fetched 517 candles
✅ EURJPY=X: fetched 517 candles
✅ GBPJPY=X: fetched 517 candles
✅ USDTRY=X: fetched 517 candles
✅ USDZAR=X: fetched 517 candles
✅ USDMXN=X: fetched 517 candles
✅ USDSEK=X: fetched 517 candles
✅ USDNOK=X: fetched 517 candles
✅ USDHUF=X: fetched 517 candles
✅ USDPLN=X: fetched 517 candles
✅ USDTHB=X: fetched 517 candles
✅ USDINR=X: fetched 517 candles
✅ USDBRL=X: fetched 517 candles

--- Fetching crypto ---
✅ BTC-USD: fetched 731 candles
✅ ETH-USD: fetched 731 candles
✅ BNB-USD: fetched 731 candles
✅ XRP-USD: fetched 731 candles
✅ SOL-USD: fetched 731 candles
✅ ADA-USD: fetched 731 candles
✅ AVAX-USD: fetched 731 candles
✅ DOT-USD: fetched 731 candles
✅ LINK-USD: fetched 731 candles
✅ LTC-USD: fetched 731 candles

In [18]:
import csv
import os

BASE_DIR = "/content/drive/MyDrive/Project 1 - Candlestick/Data/raw"

def sanitize_symbol(symbol):
    """Turn a Yahoo symbol into something safe to use as a filename.
    e.g. 'EURUSD=X' -> 'EURUSD', 'BTC-USD' stays 'BTC-USD'
    """
    return symbol.replace("=X", "").replace("/", "-")


def save_candles_to_csv(candles, symbol, market, interval, base_dir=BASE_DIR):
    clean_symbol = sanitize_symbol(symbol)
    folder = os.path.join(base_dir, market)
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, f"{clean_symbol}_{interval}.csv")

    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=["timestamp", "open", "high", "low", "close", "volume"]
        )
        writer.writeheader()
        for row in candles:
            writer.writerow(row)

    return filepath


saved_files = []
for market, symbols_dict in all_data.items():
    for symbol, candles in symbols_dict.items():
        if not candles:
            print(f"Skipping save for {symbol} -- no data fetched")
            continue
        path = save_candles_to_csv(candles, symbol, market, interval="1d")
        saved_files.append(path)
        print(f"Saved {symbol} -> {path}")

Saved EURUSD=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/EURUSD_1d.csv
Saved GBPUSD=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/GBPUSD_1d.csv
Saved USDJPY=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/USDJPY_1d.csv
Saved USDCHF=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/USDCHF_1d.csv
Saved USDCAD=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/USDCAD_1d.csv
Saved AUDUSD=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/AUDUSD_1d.csv
Saved NZDUSD=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/NZDUSD_1d.csv
Saved EURGBP=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/EURGBP_1d.csv
Saved EURJPY=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/EURJPY_1d.csv
Saved GBPJPY=X -> /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/GBPJPY_1d.csv
Saved USDTRY=X -> /content/drive/MyDrive/Project 1 - Candles

In [19]:
print(f"{'Market':<8} {'Symbol':<10} {'Candles':<10} {'File'}")
print("-" * 60)
for path in saved_files:
    # crude parse of market/symbol back out of the path, just for display
    parts = path.split(os.sep)
    market = parts[-2]
    symbol = parts[-1].replace("_1d.csv", "")
    count = len(all_data[market][
        symbol if symbol in all_data[market] else symbol + "=X"
    ]) if symbol in all_data[market] or (symbol + "=X") in all_data[market] else "?"
    print(f"{market:<8} {symbol:<10} {str(count):<10} {path}")

Market   Symbol     Candles    File
------------------------------------------------------------
forex    EURUSD     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/EURUSD_1d.csv
forex    GBPUSD     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/GBPUSD_1d.csv
forex    USDJPY     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/USDJPY_1d.csv
forex    USDCHF     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/USDCHF_1d.csv
forex    USDCAD     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/USDCAD_1d.csv
forex    AUDUSD     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/AUDUSD_1d.csv
forex    NZDUSD     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/NZDUSD_1d.csv
forex    EURGBP     517        /content/drive/MyDrive/Project 1 - Candlestick/Data/raw/forex/EURGBP_1d.csv
forex    EURJPY     517        /content/drive/M